In [ ]:
!apt-get update -qq
!apt-get install -y build-essential cmake libboost-all-dev libeigen3-dev
!git clone https://github.com/kpu/kenlm.git
!cd kenlm && mkdir build && cd build && cmake .. && make -j4

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
build-essential is already the newest version (12.9ubuntu3).
libboost-all-dev is already the newest version (1.74.0.3ubuntu7).
libeigen3-dev is already the newest version (3.4.0-2ubuntu2).
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
0 upgraded, 0 newly installed, 0 to remove and 28 not upgraded.
fatal: destination path 'kenlm' already exists and is not an empty directory.
mkdir: cannot create directory ‘build’: File exists


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls /content/kenlm/build/bin/

build_binary  fragment	       lmplz			     query
count_ngrams  interpolate      phrase_table_vocab	     streaming_example
filter	      kenlm_benchmark  probing_hash_table_benchmark


In [ ]:
import os
os.environ['PATH'] = '/content/kenlm/build/bin:' + os.environ['PATH']
!lmplz --help

Builds unpruned language models with modified Kneser-Ney smoothing.

Please cite:
@inproceedings{Heafield-estimate,
  author = {Kenneth Heafield and Ivan Pouzyrevsky and Jonathan H. Clark and Philipp Koehn},
  title = {Scalable Modified {Kneser-Ney} Language Model Estimation},
  year = {2013},
  month = {8},
  booktitle = {Proceedings of the 51st Annual Meeting of the Association for Computational Linguistics},
  address = {Sofia, Bulgaria},
  url = {http://kheafield.com/professional/edinburgh/estimate\_paper.pdf},
}

Provide the corpus on stdin.  The ARPA file will be written to stdout.  Order of
the model (-o) is the only mandatory option.  As this is an on-disk program,
setting the temporary file location (-T) and sorting memory (-S) is recommended.

Memory sizes are specified like GNU sort: a number followed by a unit character.
Valid units are % for percentage of memory (supported platforms only) and (in
increasing powers of 1024): b, K, M, G, T, P, E, Z, Y.  Default is K (*1024).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Then save output to Drive
!mkdir -p /content/drive/MyDrive/kenlm_models
# Change --output to /content/drive/MyDrive/kenlm_models/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install chardet

In [ ]:
input_file = 'unsegACTib-tok_lines_kenlm_20260103_220119.txt'
output_file = 'corpus_char_based.txt'

print("Converting to character-based...")
with open(input_file, 'r', encoding='utf-8', errors='ignore') as f_in:
    with open(output_file, 'w', encoding='utf-8') as f_out:
        for line in f_in:
            line = line.strip()
            if line:
                # Add space between each character
                char_spaced = ' '.join(line)
                f_out.write(char_spaced + '\n')

print("✓ Character-based corpus created")

Converting to character-based...
✓ Character-based corpus created


In [ ]:
import os
os.environ['PATH'] = '/content/kenlm/build/bin:' + os.environ['PATH']

!lmplz -o 5 \
    --text corpus_char_based.txt \
    --arpa model_5gram_char-tok.arpa \
    --prune 0 1 1 2 2 \
    --discount_fallback \
    --memory 80%

!build_binary model_5gram_char-tok.arpa model_5gram_char-tok.bin

=== 1/5 Counting and sorting n-grams ===
Reading /content/corpus_char_based.txt
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
Unigram tokens 715891793 types 154
=== 2/5 Calculating and sorting adjusted counts ===
Chain sizes: 1:1848 2:1061919232 3:1991098624 4:3185757696 5:4645896704
Statistics:
1 154 D1=0.259259 D2=1.84444 D3+=0.407408
2 7055/8120 D1=0.468708 D2=1.18959 D3+=1.55605
3 80969/110377 D1=0.561627 D2=1.02254 D3+=1.51725
4 319460/629265 D1=0.623291 D2=1.08228 D3+=1.5346
5 1023579/2259595 D1=0.599344 D2=1.085 D3+=1.43683
Memory estimate for binary LM:
type       kB
probing 27546 assuming -p 1.5
probing 29935 assuming -r models -p 1.5
trie     9390 without quantization
trie     4186 assuming -q 8 -b 8 quantization 
trie     8841 assuming -a 22 array pointer compression
trie     3636 assuming -a 22 -q 8 -b 8 array pointer co

In [ ]:
import json
from datetime import datetime
from pathlib import Path
import platform

# Configuration - UPDATE THESE
corpus_file = "unsegACTib_lines_kenlm_20260103_220309.txt"  # Change to your actual filename
training_time_seconds = 1800  # Estimate your training time in seconds (e.g., 1800 = 30 min)
output_dir = Path("kenlm_models")
output_dir.mkdir(exist_ok=True)

# Count lines and tokens
print("Analyzing corpus...")
line_count = 0
total_tokens = 0
with open(corpus_file, 'r') as f:
    for line in f:
        line = line.strip()
        if line:
            line_count += 1
            total_tokens += len(line.split())

# Get file sizes
arpa_size = (output_dir / "model_5gram_nontok.arpa").stat().st_size / (1024**2)
bin_size = (output_dir / "model_5gram_nontok.bin").stat().st_size / (1024**2)

# Optional: Run evaluation if you have a test file
test_file = None  # Set to "test.txt" if you have one
eval_results = None

if test_file:
    print("Running evaluation...")
    import subprocess
    import os
    os.environ['PATH'] = '/content/kenlm/build/bin:' + os.environ['PATH']

    with open(test_file, 'r') as f:
        test_text = f.read()

    result = subprocess.run(
        ['query', str(output_dir / 'model_5gram_nontok.bin')],
        input=test_text,
        capture_output=True,
        text=True
    )

    perplexity = None
    oov_rate = None

    for line in result.stdout.split('\n'):
        if 'Perplexity including OOVs' in line:
            try:
                perplexity = float(line.split('=')[1].strip())
            except:
                pass
        if 'OOV' in line and '%' in line:
            try:
                parts = line.split()
                for i, part in enumerate(parts):
                    if part == 'OOV:' and i + 1 < len(parts):
                        oov_rate = float(parts[i+1].rstrip('%'))
            except:
                pass

    if perplexity:
        eval_results = {
            'perplexity': perplexity,
            'oov_rate': oov_rate,
            'test_corpus': test_file
        }

# Create JSON report
report = {
    'timestamp': datetime.now().isoformat(),
    'system_info': {
        'environment': 'Google Colab',
        'hostname': platform.node(),
        'os': f"{platform.system()} {platform.release()}"
    },
    'model_parameters': {
        'ngram_order': 5,
        'prune_threshold': [0, 1, 1, 2, 2],
        'discount_fallback': True,
        'memory': '80%'
    },
    'corpus_statistics': {
        'line_count': line_count,
        'total_tokens': total_tokens,
        'avg_tokens_per_line': total_tokens / line_count,
        'source_files': 1,
        'combined': False
    },
    'training': {
        'training_time_seconds': training_time_seconds,
        'binary_model_path': str(output_dir / 'model_5gram_nontok.bin'),
        'binary_size_mb': bin_size
    }
}

if eval_results:
    report['evaluation'] = eval_results

with open(output_dir / 'training_report_nontok.json', 'w') as f:
    json.dump(report, f, indent=2)

# Create text report
with open(output_dir / "training_report_nontok.txt", 'w') as f:
    f.write("="*70 + "\n")
    f.write("KenLM TRAINING REPORT (Google Colab)\n")
    f.write("="*70 + "\n\n")

    f.write(f"Training Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Total Training Time: {training_time_seconds:.2f} seconds ({training_time_seconds/60:.1f} minutes)\n\n")

    f.write("-"*70 + "\n")
    f.write("SYSTEM INFORMATION\n")
    f.write("-"*70 + "\n")
    f.write("Environment: Google Colab\n")
    f.write(f"Hostname: {platform.node()}\n")
    f.write(f"OS: {platform.system()} {platform.release()}\n\n")

    f.write("-"*70 + "\n")
    f.write("MODEL PARAMETERS\n")
    f.write("-"*70 + "\n")
    f.write("N-gram Order: 5\n")
    f.write("Pruning Thresholds: [0, 1, 1, 2, 2]\n")
    f.write("Discount Fallback: True\n")
    f.write("Memory Allocation: 80%\n\n")

    f.write("-"*70 + "\n")
    f.write("CORPUS STATISTICS\n")
    f.write("-"*70 + "\n")
    f.write(f"Total Lines: {line_count:,}\n")
    f.write(f"Total Tokens: {total_tokens:,}\n")
    f.write(f"Average Tokens per Line: {total_tokens/line_count:.1f}\n")
    f.write("Source Files: 1\n")
    f.write("Files Combined: No\n\n")

    f.write("-"*70 + "\n")
    f.write("OUTPUT FILES\n")
    f.write("-"*70 + "\n")
    f.write(f"Binary Model: {output_dir / 'model_5gram_nontok.bin'}\n")
    f.write(f"Model Size: {bin_size:.2f} MB\n")
    f.write(f"ARPA Model: {output_dir / 'model_5gram_nontok.arpa'}\n")
    f.write(f"ARPA Size: {arpa_size:.2f} MB\n\n")

    if eval_results:
        f.write("-"*70 + "\n")
        f.write("EVALUATION RESULTS\n")
        f.write("-"*70 + "\n")
        f.write(f"Test Corpus: {eval_results['test_corpus']}\n")
        if eval_results.get('perplexity'):
            f.write(f"Perplexity: {eval_results['perplexity']:.2f}\n")
        if eval_results.get('oov_rate') is not None:
            f.write(f"OOV Rate: {eval_results['oov_rate']:.2f}%\n")
        f.write("\n")

    f.write("="*70 + "\n")
    f.write("END OF REPORT\n")
    f.write("="*70 + "\n")

print("✓ JSON report saved to kenlm_models/training_report_nontok.json")
print("✓ Text report saved to kenlm_models/training_report_nontok.txt")
print(f"\nModel ready! Download:")
print("  - model_5gram.bin")
print("  - training_report_nontok.txt")
print("  - training_report_nontok.json")

Analyzing corpus...
✓ JSON report saved to kenlm_models/training_report_nontok.json
✓ Text report saved to kenlm_models/training_report_nontok.txt

Model ready! Download:
  - model_5gram.bin
  - training_report_nontok.txt
  - training_report_nontok.json
